In [ ]:
# Example 1: Construct a SchemaPath from a dict using the primary entry point.
# NOTE: SchemaPath.__getitem__ resolves directly to the underlying content value
# (a plain str/dict), while the `/` operator returns a sub-SchemaPath that
# supports read_value(). Exercise both access styles.
from jsonschema_path import SchemaPath

schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer", "minimum": 0},
    },
    "required": ["name"],
}

sp = SchemaPath.from_dict(schema)
print(f"type: {type(sp).__name__}")
print(f"resolved schema type (getitem): {sp['type']}")
assert sp["type"] == "object"
assert (sp / "type").read_value() == "object"


In [ ]:
# Example 2: Navigate into nested properties and read values via SchemaPath indexing.
from jsonschema_path import SchemaPath

schema = {
    "type": "object",
    "properties": {
        "user": {
            "type": "object",
            "properties": {
                "email": {"type": "string", "format": "email"},
            },
        },
    },
}

sp = SchemaPath.from_dict(schema)
email_path = sp / "properties" / "user" / "properties" / "email"
value = email_path.read_value()
print(f"email schema: {value}")
assert value["type"] == "string"
assert value["format"] == "email"

In [ ]:
# Example 3: Iterate the keys of a SchemaPath object like a mapping.
from jsonschema_path import SchemaPath

schema = {
    "type": "object",
    "properties": {
        "a": {"type": "integer"},
        "b": {"type": "string"},
        "c": {"type": "boolean"},
    },
}

sp = SchemaPath.from_dict(schema)
props = sp / "properties"
keys = list(props.keys())
print(f"property keys: {keys}")
assert set(keys) == {"a", "b", "c"}

In [ ]:
# Example 4: Inspect the parts (JSON Pointer-like segments) of a nested SchemaPath.
from jsonschema_path import SchemaPath

schema = {
    "properties": {
        "items": {
            "type": "array",
            "items": {"type": "string"},
        },
    },
}

sp = SchemaPath.from_dict(schema)
nested = sp / "properties" / "items" / "items"
parts = list(nested.parts)
print(f"path parts: {parts}")
assert parts[-3:] == ["properties", "items", "items"]
assert nested.read_value() == {"type": "string"}

In [ ]:
# Example 5: Resolve a $ref using the referencing-backed accessor.
from jsonschema_path import SchemaPath

schema = {
    "definitions": {
        "Item": {"type": "object", "properties": {"id": {"type": "integer"}}},
    },
    "type": "object",
    "properties": {
        "item": {"$ref": "#/definitions/Item"},
    },
}

sp = SchemaPath.from_dict(schema)
item = sp / "properties" / "item"
resolved = item.read_value()
print(f"resolved $ref target: {resolved}")
assert "$ref" in resolved or resolved.get("type") == "object"